# SAM3 Tracking Head In Lumen

This notebook demonstrates the Lumen-integrated `sam3-tracking` head.

Design goals:
- use `build_head("sam3-tracking", ...)` instead of a notebook-local wrapper
- depend on the installed official `sam3` pip package, not `sam3-main/`
- keep the workflow stateful: `init_state -> add_box_prompt -> propagate`


## 1. Runtime Dependency

Make sure the official `sam3` package is installed. If not, run:

```bash
uv pip install git+https://github.com/facebookresearch/sam3.git
pip install "git+https://github.com/hyper-instrument/hyper-data.git#subdirectory=client"
pip install sqlalchemy minio "pylance>=0.20.0"
```

This notebook uses the local checkpoint `checkpoints/sam3/sam3.pt` and a local HyperData cache under `data/hyperdata/tracking-demo`. If the cache is missing, the notebook will try to pull the shared dataset from HyperData Hub automatically via `hd hub pull`.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from hyperdata import HyperData
from PIL import Image

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the Lumen repository root. "
        "Please set REPO_ROOT manually."
    )

try:
    NB_PATH = Path(__file__).resolve()
    REPO_ROOT = find_repo_root(NB_PATH)
except NameError:
    REPO_ROOT = find_repo_root(Path.cwd())

sys.path.insert(0, str(REPO_ROOT / "src"))

from lumen.data.hyperdata import HYPERDATA_AVAILABLE
from lumen.models import build_head

CHECKPOINT_PATH = REPO_ROOT / "checkpoints" / "sam3" / "sam3.pt"
TRACKING_HYPERDATA_DIR = REPO_ROOT / "data" / "hyperdata" / "tracking-demo"
HUB_IDENTIFIER = os.environ.get("LUMEN_SAM3_HYPERDATA_IDENTIFIER", "@lumen/tracking-demo")
HD_BACKEND = os.environ.get("LUMEN_HYPERDATA_BACKEND")
os.environ.setdefault("HYPERDATA_TASKS_DB", str(REPO_ROOT / "artifacts" / "hyperdata_tasks.db"))
OUTPUT_DIR = REPO_ROOT / "artifacts" / "sam3_tracking_demo"
FRAME_CACHE_DIR = OUTPUT_DIR / "hyperdata_frame_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_hd_hub_pull(identifier: str, output_dir: Path, *, backend: str | None = None) -> None:
    cmd = [sys.executable, "-m", "hyperdata._core_cli", "hub", "pull", identifier, "-o", str(output_dir)]
    if backend:
        cmd.extend(["--backend", backend])
    result = subprocess.run(cmd, cwd=REPO_ROOT, env=os.environ.copy(), capture_output=True, text=True, check=False)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"hd hub pull failed for {identifier} with exit code {result.returncode}")

def materialize_pulled_dataset(identifier: str, staging_dir: Path, dataset_dir: Path) -> None:
    pulled_root = staging_dir / identifier
    if not pulled_root.exists():
        raise RuntimeError(f"hd hub pull finished, but dataset root is missing: {pulled_root}")
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(pulled_root), str(dataset_dir))
    if staging_dir.exists():
        shutil.rmtree(staging_dir)

def ensure_local_hyperdata_dataset(dataset_dir: Path, identifier: str, required_keys: set[str], *, backend: str | None = None) -> tuple[HyperData, list[str]]:
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    needs_pull = True
    if dataset_dir.exists():
        probe = HyperData(str(dataset_dir))
        needs_pull = not list(probe.keys())
    if needs_pull:
        print(f"Local HyperData dataset missing or empty. Pulling from Hub: {identifier}")
        staging_dir = dataset_dir.parent / f".{dataset_dir.name}_pull"
        if staging_dir.exists():
            shutil.rmtree(staging_dir)
        run_hd_hub_pull(identifier, staging_dir, backend=backend)
        materialize_pulled_dataset(identifier, staging_dir, dataset_dir)
    dataset = HyperData(str(dataset_dir))
    dataset_keys = list(dataset.keys())
    missing_keys = required_keys.difference(dataset_keys)
    if missing_keys:
        raise RuntimeError(f"HyperData dataset at {dataset_dir} is missing keys: {sorted(missing_keys)}")
    return dataset, dataset_keys

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing SAM3 checkpoint: {CHECKPOINT_PATH}")
if not HYPERDATA_AVAILABLE:
    raise ImportError("HyperData is not installed. Install the thin client with: pip install 'git+https://github.com/hyper-instrument/hyper-data.git#subdirectory=client' and pip install sqlalchemy minio 'pylance>=0.20.0'")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hd, dataset_keys = ensure_local_hyperdata_dataset(TRACKING_HYPERDATA_DIR, HUB_IDENTIFIER, {"frames"}, backend=HD_BACKEND)

print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"HyperData dataset: {TRACKING_HYPERDATA_DIR}")
print(f"Hub identifier: {HUB_IDENTIFIER}")
print(f"Dataset keys: {dataset_keys}")
print(f"Frames shape: {hd['frames'].shape}")


## 2. Materialize HyperData Frames For Tracking

This notebook uses a local HyperData cache at `data/hyperdata/tracking-demo`. If the cache is missing, the setup cell automatically runs `hd hub pull` against the configured Hub identifier.

The official SAM3 tracking predictor still expects a video path or a frame directory. To keep the notebook reproducible while still using the official runtime, this cell reads the frame array from HyperData and writes a temporary JPEG frame cache that `sam3-tracking` can consume directly.


In [ ]:
def to_numpy_array(node) -> np.ndarray:
    return node.to_numpy() if hasattr(node, "to_numpy") else np.asarray(node)


def materialize_tracking_frames(dataset: HyperData, destination: Path) -> tuple[list[Path], np.ndarray]:
    dataset_keys = set(dataset.keys())
    frames = to_numpy_array(dataset["frames"])
    if frames.ndim != 4:
        raise ValueError(f"Expected tracking frames shaped (T, H, W, C), got {frames.shape}")

    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)

    if "frame_names" in dataset_keys:
        raw_names = to_numpy_array(dataset["frame_names"])
        frame_names = [str(name) for name in raw_names.tolist()]
    else:
        frame_names = [f"{index + 1:06d}.jpg" for index in range(frames.shape[0])]

    frame_paths: list[Path] = []
    for frame_name, frame in zip(frame_names, frames):
        frame_path = destination / frame_name
        Image.fromarray(frame.astype(np.uint8)).save(frame_path, format="JPEG")
        frame_paths.append(frame_path)

    return frame_paths, frames


def load_prompt_spec(dataset: HyperData, frames: np.ndarray) -> tuple[int, int, np.ndarray]:
    dataset_keys = set(dataset.keys())
    if {"init_frame_indices", "obj_ids", "init_boxes"}.issubset(dataset_keys):
        frame_idx = int(to_numpy_array(dataset["init_frame_indices"])[0])
        obj_id = int(to_numpy_array(dataset["obj_ids"])[0])
        box = to_numpy_array(dataset["init_boxes"])[0].astype(np.float32)
        return frame_idx, obj_id, box

    frame_height, frame_width = frames.shape[1:3]
    box = np.array([
        frame_width * 0.25,
        frame_height * 0.20,
        frame_width * 0.75,
        frame_height * 0.80,
    ], dtype=np.float32)
    return 0, 1, box


frame_paths, frame_array = materialize_tracking_frames(hd, FRAME_CACHE_DIR)
prompt_frame_idx, prompt_obj_id, box_xyxy = load_prompt_spec(hd, frame_array)

print(f"Materialized frames: {len(frame_paths)}")
print(f"Frame cache: {FRAME_CACHE_DIR}")
print(f"First frame: {frame_paths[0].name}")
print(f"Prompt frame: {prompt_frame_idx}, obj_id: {prompt_obj_id}")
print(f"Prompt box: {box_xyxy.tolist()}")


## 3. Build The Tracking Head Through Lumen

This now uses the registered Lumen head instead of a notebook-local wrapper.


In [ ]:
tracking_head = build_head(
    "sam3-tracking",
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
)
state = tracking_head.init_state(FRAME_CACHE_DIR)
print(f"Loaded {state['num_frames']} frames")
print(f"Original frame size: {(state['video_height'], state['video_width'])}")


## 4. Add The Stored Box Prompt

The default prompt comes from the HyperData dataset so that the whole demo remains reproducible. Replace it with your own prompt when adapting the notebook to a new tracking target.


In [ ]:
frame_height = int(state["video_height"])
frame_width = int(state["video_width"])

_ = tracking_head.add_box_prompt(
    frame_idx=prompt_frame_idx,
    obj_id=prompt_obj_id,
    box=box_xyxy,
    frame_height=frame_height,
    frame_width=frame_width,
)
print(f"Added box prompt: {box_xyxy.tolist()}")


## 5. Propagate The Mask Through The Example Video


In [ ]:
outputs = tracking_head.propagate(
    start_frame_idx=0,
    max_frame_num_to_track=len(frame_paths),
    reverse=False,
)
print(f"Tracked frames: {len(outputs)}")


In [ ]:
def to_image_mask(mask_like: torch.Tensor | np.ndarray, image_shape: tuple[int, int, int]) -> np.ndarray:
    """Convert a SAM3 mask tensor to a 2-D boolean mask aligned to an RGB image."""
    mask_np = mask_like.detach().cpu().numpy() if torch.is_tensor(mask_like) else np.asarray(mask_like)
    mask_np = np.squeeze(mask_np)
    if mask_np.ndim != 2:
        raise ValueError(f"Expected a 2-D mask after squeeze, got shape {mask_np.shape}")
    image_h, image_w = image_shape[:2]
    if mask_np.shape != (image_h, image_w):
        mask_img = Image.fromarray((mask_np > 0).astype(np.uint8) * 255)
        mask_img = mask_img.resize((image_w, image_h), resample=Image.NEAREST)
        mask_np = np.asarray(mask_img) > 0
    else:
        mask_np = mask_np > 0
    return mask_np


def overlay_mask(
    image: np.ndarray,
    mask: np.ndarray,
    color: tuple[int, int, int] = (255, 0, 0),
    alpha: float = 0.35,
) -> np.ndarray:
    base = image.astype(np.float32).copy()
    mask_bool = to_image_mask(mask, image.shape)
    color_arr = np.array(color, dtype=np.float32).reshape(1, 1, 3)
    base[mask_bool] = (1.0 - alpha) * base[mask_bool] + alpha * color_arr[0, 0]
    return base.clip(0, 255).astype(np.uint8)


sample_indices = sorted({0, len(frame_paths) // 3, 2 * len(frame_paths) // 3, len(frame_paths) - 1})
fig, axes = plt.subplots(1, len(sample_indices), figsize=(4 * len(sample_indices), 4))
if len(sample_indices) == 1:
    axes = [axes]

for axis, frame_idx in zip(axes, sample_indices):
    image = np.asarray(Image.open(frame_paths[frame_idx]).convert("RGB"))
    masks = outputs[frame_idx]["video_res_masks"]
    mask = to_image_mask(masks[0], image.shape)
    axis.imshow(overlay_mask(image, mask))
    axis.set_title(f"Frame {frame_idx}")
    axis.axis("off")

plt.tight_layout()


## 6. Optional: Export Propagated Masks

If you want to feed tracking outputs into later pseudo-labeling or analysis stages, save the propagated masks for each frame.


In [ ]:
mask_dir = OUTPUT_DIR / "masks"
mask_dir.mkdir(parents=True, exist_ok=True)

for frame_idx, payload in outputs.items():
    image = np.asarray(Image.open(frame_paths[frame_idx]).convert("RGB"))
    mask = to_image_mask(payload["video_res_masks"][0], image.shape).astype(np.uint8) * 255
    Image.fromarray(mask).save(mask_dir / f"{frame_idx:05d}.png")

print(f"Saved propagated masks to: {mask_dir}")
